# V79 - Cross-encoder chat-syllabus scores

Pre-registered plan: bi-encoder cosine (v55/v58) never beat 0.49, but cross-attention between chat and syllabus is a different relevance function the hunt has not tested. Multilingual MiniLM scores all user-module pairs zero-shot with no training, so the verdict is pure signal measurement under the standard grouped folds against word-only Ridge. Model downloads on first run. Slow on CPU by design; cache scores to disk.

In [1]:
import re
from pathlib import Path

import numpy as np
import pandas as pd
from sentence_transformers import CrossEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import Ridge
from sklearn.model_selection import StratifiedGroupKFold

SEED = 6161
N_SPLITS = 5
GATE_FLOOR = 0.002
MODEL = "cross-encoder/mmarco-mMiniLMv2-L12-H384-v1"
rng = np.random.RandomState(SEED)

CANDIDATES = [Path("/kaggle/input/intelligo-data/data"), Path("data")]
DATA = next((p for p in CANDIDATES if (p / "train_relevance.csv").exists()), None)
assert DATA is not None, "train_relevance.csv not found"
OUT = Path("/kaggle/working" if Path("/kaggle/working").exists() else "results/v79")
OUT.mkdir(parents=True, exist_ok=True)
print("DATA:", DATA.resolve())

DATA: /home/athilla/Documents/IF_ITB/Lomba/IT-TODAY/intelligo-course-recommender/data


## Zero-shot pair scores, then gate

No training touches these scores before evaluation. Chats truncate to 1000 characters per user to bound cost.

In [2]:
MODS = [f"M_{i:03d}" for i in range(1, 18)]
train = pd.read_csv(DATA / "train_relevance.csv")
test_ids = pd.read_csv(DATA / "test.csv")["user_id"].tolist()
chats = pd.read_csv(DATA / "chat_history.csv")
catalog = pd.read_csv(DATA / "modules_catalog.csv")
assert not (set(train["user_id"]) & set(test_ids))
Y = train[MODS].to_numpy(float)
n_train = len(train)
all_users = list(train["user_id"]) + test_ids
by_user = chats.groupby("user_id")["user_chat_text"].apply(list)
docs = [" ".join(str(t) for t in by_user.get(u, []))[:1000] for u in all_users]
syll = [str(catalog.set_index("module_id").loc[m, "description_and_syllabus"])[:1000] if m in set(catalog["module_id"]) else "" for m in MODS]

cache = OUT / "ce_scores.npy"
if cache.exists():
    S = np.load(cache)
    print("loaded cached scores", S.shape)
else:
    ce = CrossEncoder(MODEL, max_length=256)
    S = np.zeros((len(all_users), len(MODS)))
    for j, s in enumerate(syll):
        S[:, j] = ce.predict([(d, s) for d in docs], batch_size=64, show_progress_bar=True)
        print(f"module {j + 1}/17 scored", flush=True)
    np.save(cache, S)
S_tr = S[:n_train]

by_raw = chats.groupby("user_id")["user_chat_text"].apply(list)

def template_group(texts):
    if not texts:
        return "NO_CHAT"
    return re.sub(r"\d+", "#", " ".join(str(texts[0]).lower().split()[:6]))
fold_groups = np.array([template_group(by_raw.get(u, [])) for u in train["user_id"]])
strata = Y.argmax(1)

def ndcg5(yt, ys):
    order = np.argsort(-ys, axis=1)
    rel = np.take_along_axis(yt, order, 1)[:, :5]
    disc = 1.0 / np.log2(np.arange(2, 7))
    dcg = (rel * disc).sum(1)
    ideal = np.sort(yt, axis=1)[:, ::-1][:, :5]
    idcg = (ideal * disc).sum(1)
    return np.where(idcg > 0, dcg / np.maximum(idcg, 1e-12), 0.0)

W = TfidfVectorizer(max_features=5000, ngram_range=(1, 1), min_df=3).fit_transform(docs).toarray()
oof_w = np.zeros_like(Y)
fids = np.zeros(n_train, int)
skf = StratifiedGroupKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
for fi, (tr, va) in enumerate(skf.split(W[:n_train], strata, fold_groups)):
    for j in range(len(MODS)):
        m = Ridge(alpha=1.0)
        m.fit(W[:n_train][tr], Y[tr, j])
        oof_w[va, j] = m.predict(W[:n_train][va])
    fids[va] = fi + 1
base = ndcg5(Y, oof_w)
d = ndcg5(Y, S_tr) - base
lo, hi = np.quantile([rng.choice(d, len(d), replace=True).mean() for _ in range(2000)], [0.025, 0.975])
print(f"cross-encoder OOF={ndcg5(Y, S_tr).mean():.5f} baseline={base.mean():.5f} delta={d.mean():+.5f} CI95=[{lo:+.5f},{hi:+.5f}]")
print("ADOPT cross-encoder" if lo > GATE_FLOOR else "KEEP baseline; zero-shot cross-attention adds nothing")

loaded cached scores (5000, 17)


cross-encoder OOF=0.37933 baseline=0.62521 delta=-0.24589 CI95=[-0.25397,-0.23805]
KEEP baseline; zero-shot cross-attention adds nothing
